# Restaurant Bill Prediction

Анализ данных и предсказание среднего чека заведений общественного питания.

**Задача:** по характеристикам заведения (город, рейтинг, тип, набор фичей) предсказать средний чек.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, balanced_accuracy_score, accuracy_score

sns.set_style("whitegrid")

## 1. Загрузка данных

In [ ]:
data = pd.read_csv('data/organisations.csv')
features = pd.read_csv('data/features.csv')
rubrics = pd.read_csv('data/rubrics.csv')

print(f"Organisations: {data.shape}")
print(f"Features: {features.shape}")
print(f"Rubrics: {rubrics.shape}")

In [ ]:
data.head()

Признаки:

- `org_id` — идентификатор заведения (не используется)
- `city` — город (`msk` или `spb`)
- `average_bill` — средний чек (целевая переменная)
- `rating` — рейтинг заведения
- `rubrics_id` — типы заведения (разделённые пробелами коды)
- `features_id` — набор фичей заведения (коды через пробел)

In [ ]:
rubrics_dict = dict(zip(rubrics['rubric_id'], rubrics['rubric_name']))
features_dict = dict(zip(features['feature_id'], features['feature_name']))

rubrics_dict

## 2. Разведочный анализ данных

Смотрим на распределение таргета, пропуски и выбросы.

In [ ]:
data['average_bill'].hist(bins=50)
plt.xlabel('Average bill')
plt.ylabel('Count')
plt.title('Distribution of average bill')
plt.show()

In [ ]:
print(f"Пропусков в таргете: {data['average_bill'].isna().sum()}")
print(f"Уникальных значений таргета: {data['average_bill'].nunique()}")
print(f"Топ-5 значений:\n{data['average_bill'].value_counts().head()}")

**Наблюдения:**
- В таргете много пропусков
- Все значения кратны 500
- Есть выбросы >100 000
- Распределение сильно скошено: у ~2/3 заведений средний чек 500

In [ ]:
# Сравнение средних чеков по городам
data.boxplot(column='average_bill', by='city', figsize=(8, 5))
plt.title('Average bill by city')
plt.suptitle('')
plt.show()

In [ ]:
# Связь рейтинга и среднего чека
plt.figure(figsize=(8, 5))
plt.scatter(data['rating'], data['average_bill'], alpha=0.1, s=5)
plt.xlabel('Rating')
plt.ylabel('Average bill')
plt.title('Rating vs Average bill')
plt.show()

print(f"Корреляция: {data['average_bill'].corr(data['rating']):.4f}")

**Вывод:** корреляция между рейтингом и средним чеком практически отсутствует (~0.005).

## 3. Очистка данных

Убираем заведения с неизвестным средним чеком и выбросы (>2500).

In [ ]:
clean_data = data[(data['average_bill'].notna()) & (data['average_bill'] <= 2500)].copy()
print(f"Размер очищенного датасета: {len(clean_data)}")
print(f"Потеряно строк: {len(data) - len(clean_data)}")

In [ ]:
# Средние чеки по городам
city_means = clean_data.groupby('city')['average_bill'].mean()
print(city_means)
print(f"Разница Москва - Питер: {city_means['msk'] - city_means['spb']:.0f} руб.")

In [ ]:
# Разница между пабами и ресторанами
pub = clean_data[clean_data['rubrics_id'].str.contains('30770')]
rest = clean_data[clean_data['rubrics_id'].str.contains('30776')]
print(f"Средний чек в пабах: {pub['average_bill'].mean():.0f}")
print(f"Средний чек в ресторанах: {rest['average_bill'].mean():.0f}")
print(f"Разница: {pub['average_bill'].mean() - rest['average_bill'].mean():.0f}")

**Выводы:**
- Средний чек в Москве выше, чем в Санкт-Петербурге
- В ресторанах средний чек существенно выше, чем в пабах

## 4. Постановка задачи

Таргет принимает всего 5 значений (500, 1000, 1500, 2000, 2500), поэтому формально
задачу можно решать и как регрессию, и как классификацию. Рассмотрим обе.

**Метрики:**
- Для регрессии — RMSE
- Для классификации — balanced accuracy (обычная accuracy не подходит из-за дисбаланса классов)

In [ ]:
clean_data_train, clean_data_test = train_test_split(
    clean_data,
    stratify=clean_data['average_bill'],
    test_size=0.33,
    random_state=42
)

print(f"Train: {len(clean_data_train)}")
print(f"Test: {len(clean_data_test)}")

## 5. Бейзлайны

Простейшие модели: предсказание среднего и самого частого класса.

In [ ]:
from sklearn.base import RegressorMixin, ClassifierMixin


class MeanRegressor(RegressorMixin):
    """Predicts the mean of y_train."""
    def fit(self, X=None, y=None):
        self.mean_ = y.mean()

    def predict(self, X=None):
        return np.full(len(X), self.mean_)


class MostFrequentClassifier(ClassifierMixin):
    """Predicts the most frequent class in y_train."""
    def fit(self, X=None, y=None):
        values, counts = np.unique(y, return_counts=True)
        self.most_frequent_ = values[np.argmax(counts)]

    def predict(self, X=None):
        return np.full(len(X), self.most_frequent_)

In [ ]:
reg = MeanRegressor()
reg.fit(y=clean_data_train['average_bill'])

clf = MostFrequentClassifier()
clf.fit(y=clean_data_train['average_bill'])

y_true = clean_data_test['average_bill']

rmse_reg = np.sqrt(mean_squared_error(y_true, reg.predict(clean_data_test)))
rmse_clf = np.sqrt(mean_squared_error(y_true, clf.predict(clean_data_test)))
ba_clf = balanced_accuracy_score(y_true, clf.predict(clean_data_test))

print(f"MeanRegressor:              RMSE = {rmse_reg:.2f}")
print(f"MostFrequentClassifier:     RMSE = {rmse_clf:.2f}, BA = {ba_clf:.3f}")

## 6. Улучшение: учёт города

In [ ]:
class CityMeanRegressor(RegressorMixin):
    """Predicts mean target per city."""
    def fit(self, X=None, y=None):
        is_msk = X['city'] == 'msk'
        self.msk_mean_ = y[is_msk].mean()
        self.spb_mean_ = y[~is_msk].mean()

    def predict(self, X=None):
        is_msk = X['city'] == 'msk'
        return np.where(is_msk, self.msk_mean_, self.spb_mean_)


reg_city = CityMeanRegressor()
reg_city.fit(X=clean_data_train, y=clean_data_train['average_bill'])
rmse_city = np.sqrt(mean_squared_error(y_true, reg_city.predict(clean_data_test)))

print(f"CityMeanRegressor: RMSE = {rmse_city:.2f}")

Прирост небольшой, но он есть. MSE плохо отражает реальное качество:
предсказание 793 вместо 752 формально лучше, чем 500 вместо 1000, хотя по жизни это не так.

## 7. Feature engineering: типы заведений

В `rubrics_id` может быть несколько кодов. Сольём редкие комбинации в `other`.

In [ ]:
from collections import Counter

rubrics_counts = Counter(clean_data_train['rubrics_id'])
popular_rubrics = {r for r, c in rubrics_counts.items() if c >= 100}


def replace_rare(rubric):
    return rubric if rubric in popular_rubrics else 'other'


clean_data_train['modified_rubrics'] = clean_data_train['rubrics_id'].apply(replace_rare)
clean_data_test['modified_rubrics'] = clean_data_test['rubrics_id'].apply(replace_rare)

In [ ]:
class RubricClassifier(ClassifierMixin):
    """Predicts median target per (rubrics, city)."""
    def fit(self, X=None, y=None):
        self.global_median_ = y.median()
        self.medians_ = {}
        for (rubrics, city), group in X.assign(target=y).groupby(['modified_rubrics', 'city']):
            self.medians_[(rubrics, city)] = group['target'].median()

    def predict(self, X=None):
        predictions = []
        for i in range(len(X)):
            key = (X['modified_rubrics'].iloc[i], X['city'].iloc[i])
            predictions.append(self.medians_.get(key, self.global_median_))
        return np.array(predictions)


clf_rubric = RubricClassifier()
clf_rubric.fit(X=clean_data_train, y=clean_data_train['average_bill'])

pred_rubric = clf_rubric.predict(clean_data_test)
rmse_rubric = np.sqrt(mean_squared_error(y_true, pred_rubric))
ba_rubric = balanced_accuracy_score(y_true, pred_rubric)

print(f"RubricClassifier: RMSE = {rmse_rubric:.2f}, BA = {ba_rubric:.3f}")

## 8. Переобучение

Если объединить `rubrics_id` и `features_id` в один категориальный признак,
число категорий становится сопоставимо с числом объектов — модель переобучается.

In [ ]:
clean_data_train['modified_features'] = (
    clean_data_train['rubrics_id'] + 'q' + clean_data_train['features_id']
)

test_concat = clean_data_test['rubrics_id'] + 'q' + clean_data_test['features_id']
train_features_set = set(clean_data_train['modified_features'])

clean_data_test['modified_features'] = np.where(
    test_concat.isin(train_features_set),
    test_concat,
    'other'
)

In [ ]:
class FeaturesMedianClassifier(ClassifierMixin):
    """Predicts median per concatenated (rubrics + features)."""
    def fit(self, X=None, y=None):
        self.global_median_ = y.median()
        self.medians_ = {}
        for feature, group in X.assign(target=y).groupby('modified_features'):
            self.medians_[feature] = group['target'].median()

    def predict(self, X=None):
        predictions = []
        for i in range(len(X)):
            key = X['modified_features'].iloc[i]
            predictions.append(self.medians_.get(key, self.global_median_))
        return np.array(predictions)


clf_features = FeaturesMedianClassifier()
clf_features.fit(X=clean_data_train, y=clean_data_train['average_bill'])

pred_train = clf_features.predict(clean_data_train)
pred_test = clf_features.predict(clean_data_test)

rmse_train = np.sqrt(mean_squared_error(clean_data_train['average_bill'], pred_train))
rmse_test = np.sqrt(mean_squared_error(y_true, pred_test))

ba_train = balanced_accuracy_score(clean_data_train['average_bill'], pred_train)
ba_test = balanced_accuracy_score(y_true, pred_test)

print(f"FeaturesMedianClassifier (train): RMSE = {rmse_train:.2f}, BA = {ba_train:.3f}")
print(f"FeaturesMedianClassifier (test):  RMSE = {rmse_test:.2f}, BA = {ba_test:.3f}")

**Вывод:** явное переобучение. На трейне RMSE ~32, на тесте ~514.
Чрезмерная сложность модели не помогает.

## 9. CatBoost на разреженной матрице

Превратим категориальные признаки в one-hot и обучим градиентный бустинг.

In [ ]:
from scipy.sparse import coo_matrix


all_rubrics = []
for s in clean_data_train['rubrics_id']:
    all_rubrics.extend(str(s).split())
train_rubrics = set(all_rubrics)

all_features = []
for s in clean_data_train['features_id']:
    all_features.extend(str(s).split())
train_features = set(all_features)

column_names = ['city', 'rating', 'feature_other']
column_names += ['rubric_' + r for r in sorted(train_rubrics)]
column_names += ['feature_' + f for f in sorted(train_features)]

column_number = {name: i for i, name in enumerate(column_names)}
total_columns = len(column_names)


def build_arrays(X):
    rows, cols, values = [], [], []
    for i in range(len(X)):
        row = X.iloc[i]

        if row['city'] == 'msk':
            rows.append(i); cols.append(column_number['city']); values.append(1.0)

        rows.append(i); cols.append(column_number['rating']); values.append(float(row['rating']))

        for code in str(row['rubrics_id']).split():
            name = 'rubric_' + code
            if name in column_number:
                rows.append(i); cols.append(column_number[name]); values.append(1.0)

        unknown_count = 0
        for code in str(row['features_id']).split():
            name = 'feature_' + code
            if name in column_number:
                rows.append(i); cols.append(column_number[name]); values.append(1.0)
            else:
                unknown_count += 1

        if unknown_count > 0:
            rows.append(i); cols.append(column_number['feature_other']); values.append(float(unknown_count))

    return rows, cols, values


train_rows, train_cols, train_values = build_arrays(clean_data_train)
test_rows, test_cols, test_values = build_arrays(clean_data_test)

sparse_data_train = coo_matrix(
    (np.array(train_values), (np.array(train_rows), np.array(train_cols))),
    shape=(len(clean_data_train), total_columns)
).tocsr()

sparse_data_test = coo_matrix(
    (np.array(test_values), (np.array(test_rows), np.array(test_cols))),
    shape=(len(clean_data_test), total_columns)
).tocsr()

print(f"Train matrix: {sparse_data_train.shape}")
print(f"Test matrix:  {sparse_data_test.shape}")

In [ ]:
from catboost import CatBoostClassifier

clf_cb = CatBoostClassifier(verbose=False, random_state=42)
clf_cb.fit(sparse_data_train, clean_data_train['average_bill'])

pred_cb = clf_cb.predict(sparse_data_test)
ba_cb = balanced_accuracy_score(y_true, pred_cb)

print(f"CatBoost: BA = {ba_cb:.3f}")

CatBoost показывает BA ~0.36, что немного лучше базового классификатора (~0.20),
но всё ещё далеко от идеала. Это ещё раз подтверждает: **без хороших данных даже
самая мощная модель не вытянет задачу.**

## 10. Итоговое сравнение моделей

In [ ]:
results = pd.DataFrame([
    {'Model': 'MeanRegressor',              'Type': 'Regression',     'RMSE': rmse_reg,     'BA': None},
    {'Model': 'MostFrequentClassifier',     'Type': 'Classification', 'RMSE': rmse_clf,     'BA': ba_clf},
    {'Model': 'CityMeanRegressor',          'Type': 'Regression',     'RMSE': rmse_city,    'BA': None},
    {'Model': 'RubricClassifier',           'Type': 'Classification', 'RMSE': rmse_rubric,  'BA': ba_rubric},
    {'Model': 'FeaturesMedianClassifier',   'Type': 'Classification', 'RMSE': rmse_test,    'BA': ba_test},
    {'Model': 'CatBoost',                   'Type': 'Classification', 'RMSE': None,         'BA': ba_cb},
])

results = results.round(3)
print(results.to_string(index=False))
results

## Выводы

1. **Бейзлайны** дают слабые, но осмысленные результаты: RMSE ~448–515, BA ~0.2.
2. **Учёт города** слегка улучшает RMSE (~445), но не меняет BA.
3. **Feature engineering по рубрикам** даёт наибольший прирост: RMSE ~394, BA ~0.31.
4. **Слишком сложная модель** (объединение рубрик и фичей) переобучается: RMSE на трейне ~32, на тесте ~514.
5. **CatBoost** на разреженной матрице показывает BA ~0.36 — лучший результат среди классификаторов,
   но задача всё ещё далека от идеального решения.

Главный вывод: **качество данных важнее выбора модели**. Даже градиентный бустинг не компенсирует
слабую информативность признаков.

## Что можно улучшить

- Собрать больше фичей о заведениях (меню, цены, отзывы)
- Попробовать target encoding вместо one-hot
- Использовать кросс-валидацию для более честной оценки
- Обучить регрессию вместо классификации и сравнить RMSE